In [1]:
# Add this at the VERY TOP of Cell 2, before any imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [2]:
# 1: Set Up & Utilities

In [3]:
import os, time, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")
 
# 🌱 Set seed globally
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
 
# 🌱 Multi-seed wrapper
def run_with_seeds(train_fn, seeds=[42, 123, 7], **kwargs):
    metrics = {"acc": [], "f1": [], "train_time": [], "inf_time": []}
    raw_results = []
 
    for s in seeds:
        set_seed(s)
        res = train_fn(seed=s, **kwargs)
        raw_results.append({"seed": s, **res})
 
        for k in metrics:
            if f"test_{k}" in res:
                metrics[k].append(res[f"test_{k}"])
            elif k in res:
                metrics[k].append(res[k])
            else:
                raise KeyError(f"Missing metric '{k}' in result for seed {s}")
 
    summary = {
        k: {
            "mean": float(np.mean(v)),
            "std": float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
        }
        for k, v in metrics.items()
    }
 
    return {"summary": summary, "raw_results": raw_results}
 
# 📏 Text Length & Truncation Analysis
def analyze_lengths(texts, tokenizer, max_len=512):
    lens = [len(tokenizer.encode(t, truncation=False)) for t in texts]
    stats = {
        "mean": float(np.mean(lens)),
        "median": float(np.median(lens)),
        "p95": float(np.percentile(lens, 95)),
        "max": int(np.max(lens)),
        "truncated": int(sum(l > max_len for l in lens)),
        "total": int(len(lens)),
        "truncated_pct": float(100 * sum(l > max_len for l in lens) / len(lens))
    }
 
    print(
        f"📊 Length Stats (tokens) | Mean: {stats['mean']:.1f} | "
        f"Median: {stats['median']:.1f} | P95: {stats['p95']:.1f} | Max: {stats['max']}"
    )
    print(
        f"🔪 {stats['truncated']}/{stats['total']} "
        f"({stats['truncated_pct']:.1f}%) exceed {max_len} tokens and will be truncated."
    )
    return stats
 
# ⚖️ Class Imbalance Check
def check_distribution(y, names):
    from collections import Counter
    cnt = Counter(y)
    total = len(y)
    rows = []
    for k, v in sorted(cnt.items()):
        rows.append({
            "class_id": k,
            "class_name": names[k],
            "count": v,
            "pct": 100 * v / total
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


In [4]:
# 2: Load & Split Datasets

In [5]:
# 2A: Load & Split Newsgroups (via Kaggle Hub)
import os
import re
import kagglehub
from sklearn.model_selection import train_test_split

print("📥 Loading 20 Newsgroups from Kaggle...")

# Download dataset from Kaggle (no authentication required for public datasets)
base_path = kagglehub.dataset_download("crawford/20-newsgroups")

# Find all .txt files (each represents a newsgroup category)
txt_files = sorted([f for f in os.listdir(base_path) if f.endswith('.txt')])
if not txt_files:
    raise FileNotFoundError("No .txt files found in Kaggle download.")

# Create class names and label mapping
class_names = [f.replace('.txt', '') for f in txt_files]
label_map = {name: i for i, name in enumerate(class_names)}
print(f"📂 Found {len(class_names)} newsgroup files")

all_texts = []
all_labels = []

for cls in class_names:
    file_path = os.path.join(base_path, f"{cls}.txt")
    if not os.path.exists(file_path):
        continue
    
    with open(file_path, 'r', encoding='latin-1', errors='ignore') as f:
        raw = f.read()
    
    if not raw.strip():
        print(f"⚠️ {cls}.txt is empty. Skipping.")
        continue
    
    # Strategy 1: Split by double newline (handles \n\n and \r\n\r\n)
    docs = re.split(r'\r?\n\r?\n', raw)
    
    # Strategy 2: If Strategy 1 fails (<5 docs), split by Usenet header markers
    if len(docs) < 5:
        docs = re.split(r'(?=^From:|^Subject:|^Article-I\.D\.|^Path:)', raw, flags=re.MULTILINE)
    
    # Strategy 3: Fallback to single newline if still too few
    if len(docs) < 5:
        docs = raw.split('\n')
    
    docs_found = 0
    for doc in docs:
        doc = doc.strip()
        if len(doc) < 50:  # Skip very short documents
            continue
        
        # Light cleaning: remove header lines, keep body
        lines = doc.split('\n')
        cleaned = []
        in_header = True
        
        for line in lines:
            line_stripped = line.strip()
            
            if in_header:
                # Skip standard Usenet header fields
                if re.match(r'^(From|Subject|Organization|Date|Reply-To|Lines|Distribution|Keywords|Article-I\.D\.|NNTP-Posting-Host|Path|Message-ID|X-Newsreader|X-Trace|Posted|Followup-To):', line_stripped, re.I):
                    continue
                if line_stripped == '':
                    in_header = False
                    continue
                continue
            
            # Skip quotes/signatures
            if line_stripped.startswith('>') or line_stripped.startswith('|') or line_stripped.startswith('---') or line_stripped.startswith('___'):
                continue
            
            if line_stripped:
                cleaned.append(line_stripped)
        
        final_text = ' '.join(cleaned)
        if len(final_text) > 60:  # Keep only meaningful documents
            all_texts.append(final_text)
            all_labels.append(label_map[cls])
            docs_found += 1
    
    print(f"   📄 {cls}: {docs_found} docs loaded")

print(f"\n✅ Total 20NG documents: {len(all_texts)}")

if len(all_texts) == 0:
    # Fallback: print exact file stats so we can fix it in 1 message
    print("❌ Still zero documents. Printing file diagnostics:")
    print(f"   File: {file_path}")
    print(f"   Size: {len(raw)} bytes")
    print(f"   First 300 chars:\n{raw[:300]}")
    raise SystemExit("Reply with the diagnostics above so I can give you a 1-line fix.")

# Stratified Split: 80% Train, 10% Val, 10% Test
print("🔀 Splitting data (80/10/10)...")
X_train, X_temp, y_train, y_temp = train_test_split(
    all_texts, all_labels, 
    test_size=0.2, 
    random_state=42, 
    stratify=all_labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.5, 
    random_state=42, 
    stratify=y_temp
)

news_train = (X_train, y_train)
news_val = (X_val, y_val)
news_test = (X_test, y_test)
news_names = class_names

print(f"\n📊 20 Newsgroups Ready:")
print(f"   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"   Sample text: '{X_train[0][:100]}...'")
print(f"   Sample label: {news_names[y_train[0]]}")

📥 Loading 20 Newsgroups from Kaggle...
📂 Found 20 newsgroup files
   📄 alt.atheism: 258 docs loaded
   📄 comp.graphics: 157 docs loaded
   📄 comp.os.ms-windows.misc: 140 docs loaded
   📄 comp.sys.ibm.pc.hardware: 120 docs loaded
   📄 comp.sys.mac.hardware: 142 docs loaded
   📄 comp.windows.x: 164 docs loaded
   📄 misc.forsale: 194 docs loaded
   📄 rec.autos: 90 docs loaded
   📄 rec.motorcycles: 118 docs loaded
   📄 rec.sport.baseball: 56 docs loaded
   📄 rec.sport.hockey: 204 docs loaded
   📄 sci.crypt: 228 docs loaded
   📄 sci.electronics: 114 docs loaded
   📄 sci.med: 206 docs loaded
   📄 sci.space: 128 docs loaded
   📄 soc.religion.christian: 138 docs loaded
   📄 talk.politics.guns: 170 docs loaded
   📄 talk.politics.mideast: 190 docs loaded
   📄 talk.politics.misc: 124 docs loaded
   📄 talk.religion.misc: 112 docs loaded

✅ Total 20NG documents: 3053
🔀 Splitting data (80/10/10)...

📊 20 Newsgroups Ready:
   Train: 2442 | Val: 305 | Test: 306
   Sample text: 'v)    To look into the 

In [6]:
# 2B: Load & Split TweeEval (Irony)

In [7]:
import os
import glob
import kagglehub
import pandas as pd

print("📥 Loading TweetEval from Kaggle...")
path = kagglehub.dataset_download("thedevastator/tweeteval-a-multi-task-classification-benchmark")
print(f"📁 Dataset downloaded to: {path}")

# 1. Find all irony CSVs
all_csvs = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)
irony_files = [f for f in all_csvs if "irony" in f.lower()]

if len(irony_files) < 3:
    raise ValueError(f"Could not find all 3 irony split files. Found: {[os.path.basename(f) for f in irony_files]}")

# 2. Robustly map split names to file paths
split_paths = {}
for f in irony_files:
    basename = os.path.basename(f).lower()
    if 'train' in basename and 'val' not in basename:
        split_paths['train'] = f
    elif 'validation' in basename or 'val' in basename:
        split_paths['validation'] = f
    elif 'test' in basename:
        split_paths['test'] = f

print(f"📄 Mapped splits: {list(split_paths.keys())}")

# 3. Load each split separately 
df_train = pd.read_csv(split_paths['train'])
df_val   = pd.read_csv(split_paths['validation'])
df_test  = pd.read_csv(split_paths['test'])

# 4. Identify columns dynamically
text_col = next((c for c in df_train.columns if c.lower() in ['text', 'tweet', 'sentence']), df_train.columns[0])
label_col = next((c for c in df_train.columns if c.lower() in ['label', 'class', 'target']), df_train.columns[1])
print(f"📊 Using column '{text_col}' for text and '{label_col}' for labels.")

tweet_names = ["non_ironic", "ironic"]

# 5. Clean & package into tuples (X, y)
def clean_split(df):
    df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)
    texts = df[text_col].astype(str).tolist()
    labels = df[label_col].astype(int).tolist()
    return (texts, labels)

tweet_train = clean_split(df_train)
tweet_val   = clean_split(df_val)
tweet_test  = clean_split(df_test)

print(f"\n✅ TweetEval Ready (Standard Splits):")
print(f"   Train: {len(tweet_train[0])} | Val: {len(tweet_val[0])} | Test: {len(tweet_test[0])}")
print(f"   Sample text: '{tweet_train[0][0][:80]}...'")
print(f"   Sample label: {tweet_names[tweet_train[1][0]]} (class {tweet_train[1][0]})")

📥 Loading TweetEval from Kaggle...
📁 Dataset downloaded to: /Users/fartingspirit/.cache/kagglehub/datasets/thedevastator/tweeteval-a-multi-task-classification-benchmark/versions/2
📄 Mapped splits: ['test', 'validation', 'train']
📊 Using column 'text' for text and 'label' for labels.

✅ TweetEval Ready (Standard Splits):
   Train: 2862 | Val: 955 | Test: 784
   Sample text: 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of m...'
   Sample label: ironic (class 1)


In [8]:
# ================= DIAGNOSTICS & VALIDATION =================
from transformers import AutoTokenizer
print("📊 Dataset Diagnostics & Leakage Checks")

# 1. Check Class Distribution (Class Imbalance Check)
print("\n1️⃣ Class Distributions:")
print("20 Newsgroups:")
check_distribution(news_train[1], news_names)
print("\nTweetEval Irony:")
check_distribution(tweet_train[1], tweet_names)

# 2. Check Token Lengths 
print("\n2️⃣ Text Length Analysis (for Report):")
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Reuse tokenizer to save time
analyze_lengths(news_train[0], tok, max_len=512)
analyze_lengths(tweet_train[0], tok, max_len=512)

# 3. Verify 20NG Leakage Cleaning
print("\n3️⃣ 20NG Leakage Verification (First 200 chars of first doc):")
print(news_train[0][0][:200])

📊 Dataset Diagnostics & Leakage Checks

1️⃣ Class Distributions:
20 Newsgroups:
 class_id               class_name  count      pct
        0              alt.atheism    206 8.435708
        1            comp.graphics    126 5.159705
        2  comp.os.ms-windows.misc    112 4.586405
        3 comp.sys.ibm.pc.hardware     96 3.931204
        4    comp.sys.mac.hardware    114 4.668305
        5           comp.windows.x    131 5.364455
        6             misc.forsale    155 6.347256
        7                rec.autos     72 2.948403
        8          rec.motorcycles     95 3.890254
        9       rec.sport.baseball     45 1.842752
       10         rec.sport.hockey    163 6.674857
       11                sci.crypt    182 7.452907
       12          sci.electronics     91 3.726454
       13                  sci.med    165 6.756757
       14                sci.space    102 4.176904
       15   soc.religion.christian    110 4.504505
       16       talk.politics.guns    136 5.569206
  

Token indices sequence length is longer than the specified maximum sequence length for this model (1017 > 512). Running this sequence through the model will result in indexing errors


📊 Length Stats (tokens) | Mean: 221.3 | Median: 83.0 | P95: 639.0 | Max: 9397
🔪 168/2442 (6.9%) exceed 512 tokens and will be truncated.
📊 Length Stats (tokens) | Mean: 22.9 | Median: 22.0 | P95: 38.0 | Max: 294
🔪 0/2862 (0.0%) exceed 512 tokens and will be truncated.

3️⃣ 20NG Leakage Verification (First 200 chars of first doc):
v)    To look into the origin and teachings of all religions in general and of Islam and Ahmadiyya Muslim Movement in par- ticular, and to use the commonality of origin to foster better understanding 


In [9]:
# 3: Classical Models (TF-IDR + LR / SVM)

In [10]:
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

def train_classical(X_train, y_train, X_val, y_val, X_test, y_test, model_type="lr", c_values=[0.1, 1.0, 10.0], seed=42):
    np.random.seed(seed)
    best_model = None
    best_val_f1 = -1.0
    best_C = None
    
    for C in c_values:
        if model_type == "lr":
            clf = LogisticRegression(C=C, max_iter=1000, random_state=seed, n_jobs=-1, class_weight='balanced')
        else:
            clf = LinearSVC(C=C, max_iter=2000, random_state=seed, dual=False, class_weight='balanced')
            
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)), 
            ('clf', clf)
        ])
        
        t0 = time.time()
        pipe.fit(X_train, y_train)
        train_time = time.time() - t0
        
        y_val_pred = pipe.predict(X_val)
        val_f1 = f1_score(y_val, y_val_pred, average='macro')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_C = C
            best_model = pipe
            
    # Final test evaluation
    t0 = time.time()
    y_test_pred = best_model.predict(X_test)
    inf_time_per_sample = (time.time() - t0) / len(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    # Per-class F1 extraction (Rubric requirement)
    report = classification_report(y_test, y_test_pred, output_dict=True)
    cls_f1 = {k: v['f1-score'] for k, v in report.items() if k not in ['accuracy', 'macro avg', 'weighted avg']}
    best_cls = max(cls_f1, key=cls_f1.get) if cls_f1 else None
    worst_cls = min(cls_f1, key=cls_f1.get) if cls_f1 else None
    
    return {
        'best_C': best_C, 'val_f1': best_val_f1,
        'test_acc': test_acc, 'test_f1': test_f1,
        'train_time': train_time, 'inf_time': inf_time_per_sample,
        'best_class': best_cls, 'best_class_f1': cls_f1.get(best_cls, 0.0),
        'worst_class': worst_cls, 'worst_class_f1': cls_f1.get(worst_cls, 0.0),
        'model': best_model
    }

# ================= RUN ON 20 NEWGROUPS =================
print("🚀 Training Classical Models on 20 Newsgroups...")

news_lr = run_with_seeds(train_classical, seeds=[42, 123, 7],
                         X_train=news_train[0], y_train=news_train[1],
                         X_val=news_val[0], y_val=news_val[1],
                         X_test=news_test[0], y_test=news_test[1],
                         model_type="lr")

news_svm = run_with_seeds(train_classical, seeds=[42, 123, 7],
                          X_train=news_train[0], y_train=news_train[1],
                          X_val=news_val[0], y_val=news_val[1],
                          X_test=news_test[0], y_test=news_test[1],
                          model_type="svm")

print("\n📊 20NG Results:")
print(f"   LR : Acc={news_lr['summary']['acc']['mean']:.4f} ± {news_lr['summary']['acc']['std']:.4f} | F1={news_lr['summary']['f1']['mean']:.4f} ± {news_lr['summary']['f1']['std']:.4f} | Train={news_lr['summary']['train_time']['mean']:.2f}s")
# Added best/worst class print
lr_raw = news_lr['raw_results'][0]
print(f"      🏆 Best Class: {lr_raw['best_class']} ({lr_raw['best_class_f1']:.3f}) | ⚠️ Worst: {lr_raw['worst_class']}")

print(f"   SVM: Acc={news_svm['summary']['acc']['mean']:.4f} ± {news_svm['summary']['acc']['std']:.4f} | F1={news_svm['summary']['f1']['mean']:.4f} ± {news_svm['summary']['f1']['std']:.4f} | Train={news_svm['summary']['train_time']['mean']:.2f}s")

# ================= RUN ON TWEETEVAL =================
print("\n🚀 Training Classical Models on TweetEval...")

tweet_lr = run_with_seeds(train_classical, seeds=[42, 123, 7],
                          X_train=tweet_train[0], y_train=tweet_train[1],
                          X_val=tweet_val[0], y_val=tweet_val[1],
                          X_test=tweet_test[0], y_test=tweet_test[1],
                          model_type="lr")

tweet_svm = run_with_seeds(train_classical, seeds=[42, 123, 7],
                           X_train=tweet_train[0], y_train=tweet_train[1],
                           X_val=tweet_val[0], y_val=tweet_val[1],
                           X_test=tweet_test[0], y_test=tweet_test[1],
                           model_type="svm")

print("\n📊 TweetEval Results:")
print(f"   LR : Acc={tweet_lr['summary']['acc']['mean']:.4f} ± {tweet_lr['summary']['acc']['std']:.4f} | F1={tweet_lr['summary']['f1']['mean']:.4f} ± {tweet_lr['summary']['f1']['std']:.4f} | Train={tweet_lr['summary']['train_time']['mean']:.2f}s")
print(f"   SVM: Acc={tweet_svm['summary']['acc']['mean']:.4f} ± {tweet_svm['summary']['acc']['std']:.4f} | F1={tweet_svm['summary']['f1']['mean']:.4f} ± {tweet_svm['summary']['f1']['std']:.4f} | Train={tweet_svm['summary']['train_time']['mean']:.2f}s")

🚀 Training Classical Models on 20 Newsgroups...

📊 20NG Results:
   LR : Acc=0.9020 ± 0.0000 | F1=0.8888 ± 0.0000 | Train=0.98s
      🏆 Best Class: 18 (1.000) | ⚠️ Worst: 19
   SVM: Acc=0.9052 ± 0.0000 | F1=0.8931 ± 0.0000 | Train=1.27s

🚀 Training Classical Models on TweetEval...

📊 TweetEval Results:
   LR : Acc=0.6531 ± 0.0000 | F1=0.6426 ± 0.0000 | Train=0.05s
   SVM: Acc=0.6518 ± 0.0000 | F1=0.6415 ± 0.0000 | Train=0.06s


In [11]:
# 4: BiLSTM (Tier 2 Neural Model)

In [12]:
import time, numpy as np, torch, torch.nn as nn, torch.optim as optim
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"🖥️ Using device: {device}")

class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=256):
        self.labels = list(labels)
        self.max_len = max_len
        self.vocab = vocab
        self.texts = [self.text_to_indices(t) for t in texts]
    def text_to_indices(self, text):
        tokens = text.split()
        ids = [self.vocab.get(w, 1) for w in tokens[:self.max_len]]
        return torch.tensor(ids if ids else [1], dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.texts[idx], int(self.labels[idx])

def collate_batch(batch):
    texts, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in texts], dtype=torch.long)
    padded = pad_sequence(texts, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
    def forward(self, x, lengths):
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.bilstm(packed)
        h = torch.cat((h[-2], h[-1]), dim=1)
        return self.fc(self.dropout(h))

def train_bilstm_tuned(X_train, y_train, X_val, y_val, X_test, y_test, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    
    counter = Counter(w for text in X_train for w in text.split())
    vocab = {"<pad>": 0, "<unk>": 1}
    for word, _ in counter.most_common(15000): vocab[word] = len(vocab)
    num_labels = len(np.unique(y_train))
    
    unique, counts = np.unique(y_train, return_counts=True)
    class_weights = torch.tensor(1.0 / counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    configs = [(0.001, 3), (0.002, 5), (0.005, 3)]
    best_val_f1, best_params, best_state = -1.0, None, None
    t0_total = time.time()
    
    print(f"   🌱 Seed {seed}: Starting minimal HP search ({len(configs)} configs)...")
    for i, (lr, epochs) in enumerate(configs):
        train_ds = TextDataset(X_train, y_train, vocab)
        val_ds   = TextDataset(X_val, y_val, vocab)
        train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_batch)
        val_dl   = DataLoader(val_ds, batch_size=128, shuffle=False, collate_fn=collate_batch)
        
        model = BiLSTMClassifier(len(vocab), 100, 128, num_labels).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        for _ in range(epochs):
            model.train()
            for bx, blen, by in train_dl:
                bx, blen, by = bx.to(device), blen.to(device), by.to(device)
                optimizer.zero_grad()
                loss = criterion(model(bx, blen), by)
                loss.backward()
                optimizer.step()
        
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for bx, blen, by in val_dl:
                bx, blen, by = bx.to(device), blen.to(device), by.to(device)
                preds = model(bx, blen).argmax(dim=1)
                val_preds.extend(preds.cpu().tolist())
                val_true.extend(by.cpu().tolist())
        val_f1 = f1_score(val_true, val_preds, average="macro")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_params = (lr, epochs)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"      ✅ Config {i+1}/{len(configs)} | lr={lr}, epochs={epochs} | Val F1: {val_f1:.4f}")

    test_ds = TextDataset(X_test, y_test, vocab)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False, collate_fn=collate_batch)
    model = BiLSTMClassifier(len(vocab), 100, 128, num_labels).to(device)
    model.load_state_dict(best_state)
    model.eval()
    
    t0_test = time.time()
    test_preds, test_true = [], []
    with torch.no_grad():
        for bx, blen, by in test_dl:
            bx, blen, by = bx.to(device), blen.to(device), by.to(device)
            preds = model(bx, blen).argmax(dim=1)
            test_preds.extend(preds.cpu().tolist())
            test_true.extend(by.cpu().tolist())
    inf_time = (time.time() - t0_test) / len(test_true)
    train_time = time.time() - t0_total
    
    # ✅ ADDED: Save predictions for Cells 7 & 8
    return {
        "acc": accuracy_score(test_true, test_preds),
        "f1": f1_score(test_true, test_preds, average="macro"),
        "train_time": train_time,
        "inf_time": inf_time,
        "best_lr": best_params[0],
        "best_epoch": best_params[1],
        "preds": test_preds,
        "y_true": test_true
    }

# ================= EXECUTION =================
print("🚀 Training BiLSTM (Tier 2 Neural) with 3 Seeds & Minimal Grid...")
news_bilstm = run_with_seeds(train_bilstm_tuned, seeds=[42, 123, 7], 
    X_train=news_train[0], y_train=news_train[1], X_val=news_val[0], y_val=news_val[1], 
    X_test=news_test[0], y_test=news_test[1])

tweet_bilstm = run_with_seeds(train_bilstm_tuned, seeds=[42, 123, 7], 
    X_train=tweet_train[0], y_train=tweet_train[1], X_val=tweet_val[0], y_val=tweet_val[1], 
    X_test=tweet_test[0], y_test=tweet_test[1])

print(
    f"\n📊 20NG BiLSTM: "
    f"Acc={news_bilstm['summary']['acc']['mean']:.4f}±{news_bilstm['summary']['acc']['std']:.4f} | "
    f"F1={news_bilstm['summary']['f1']['mean']:.4f}±{news_bilstm['summary']['f1']['std']:.4f} | "
    f"Train={news_bilstm['summary']['train_time']['mean']:.2f}±{news_bilstm['summary']['train_time']['std']:.2f}s | "
    f"Inf={news_bilstm['summary']['inf_time']['mean']:.6f}±{news_bilstm['summary']['inf_time']['std']:.6f}s/sample"
)

print(
    f"📊 TweetEval BiLSTM: "
    f"Acc={tweet_bilstm['summary']['acc']['mean']:.4f}±{tweet_bilstm['summary']['acc']['std']:.4f} | "
    f"F1={tweet_bilstm['summary']['f1']['mean']:.4f}±{tweet_bilstm['summary']['f1']['std']:.4f} | "
    f"Train={tweet_bilstm['summary']['train_time']['mean']:.2f}±{tweet_bilstm['summary']['train_time']['std']:.2f}s | "
    f"Inf={tweet_bilstm['summary']['inf_time']['mean']:.6f}±{tweet_bilstm['summary']['inf_time']['std']:.6f}s/sample"
)


🖥️ Using device: mps
🚀 Training BiLSTM (Tier 2 Neural) with 3 Seeds & Minimal Grid...
   🌱 Seed 42: Starting minimal HP search (3 configs)...
      ✅ Config 1/3 | lr=0.001, epochs=3 | Val F1: 0.3900
      ✅ Config 2/3 | lr=0.002, epochs=5 | Val F1: 0.8103
      ✅ Config 3/3 | lr=0.005, epochs=3 | Val F1: 0.8256
   🌱 Seed 123: Starting minimal HP search (3 configs)...
      ✅ Config 1/3 | lr=0.001, epochs=3 | Val F1: 0.3734
      ✅ Config 2/3 | lr=0.002, epochs=5 | Val F1: 0.8383
      ✅ Config 3/3 | lr=0.005, epochs=3 | Val F1: 0.7890
   🌱 Seed 7: Starting minimal HP search (3 configs)...
      ✅ Config 1/3 | lr=0.001, epochs=3 | Val F1: 0.4045
      ✅ Config 2/3 | lr=0.002, epochs=5 | Val F1: 0.8170
      ✅ Config 3/3 | lr=0.005, epochs=3 | Val F1: 0.8202
   🌱 Seed 42: Starting minimal HP search (3 configs)...
      ✅ Config 1/3 | lr=0.001, epochs=3 | Val F1: 0.5700
      ✅ Config 2/3 | lr=0.002, epochs=5 | Val F1: 0.6104
      ✅ Config 3/3 | lr=0.005, epochs=3 | Val F1: 0.5824
   🌱 S

In [13]:
# Set-Up for Loading Distilbert (Tier 3 Neural Model) and Roberta (Tier 3 Neural Model) Data

In [14]:
import os
import pickle
import numpy as np
import pandas as pd

RESULTS_DIR = "results"

DISTIL_PATHS = [
    os.path.join(RESULTS_DIR, "all_results_distil_seed42.pkl"),
    os.path.join(RESULTS_DIR, "all_results_distil_seed123.pkl"),
    os.path.join(RESULTS_DIR, "all_results_distil_seed7.pkl"),
]
ROBERTA_PATH = os.path.join(RESULTS_DIR, "all_results_roberta.pkl")

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def summarise_runs(runs):
    def mean_std(key):
        vals = [r[key] for r in runs]
        return {"mean": float(np.mean(vals)), "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0}
    return {
        "acc": mean_std("test_acc"),
        "f1": mean_std("test_f1"),
        "train_time": mean_std("train_time"),
        "inf_time": mean_std("inf_time"),
    }

# Merge DistilBERT seed files into one notebook-friendly dict
distil_merged = {}
distil_runs = {"20NG_distilbert-base-uncased": [], "TweetEval_distilbert-base-uncased": []}

for path in DISTIL_PATHS:
    obj = load_pickle(path)
    for key, val in obj.items():
        if key.startswith("20NG_distilbert-base-uncased"):
            distil_runs["20NG_distilbert-base-uncased"].append(val)
        elif key.startswith("TweetEval_distilbert-base-uncased"):
            distil_runs["TweetEval_distilbert-base-uncased"].append(val)

for key, runs in distil_runs.items():
    distil_merged[key] = {
        "summary": summarise_runs(runs),
        "raw_results": runs,
    }

roberta_results = load_pickle(ROBERTA_PATH)

all_results = {}
all_results.update(distil_merged)
all_results.update(roberta_results)

print(all_results.keys())


dict_keys(['20NG_distilbert-base-uncased', 'TweetEval_distilbert-base-uncased', '20NG_roberta-base', 'TweetEval_roberta-base'])


In [15]:
# Display the results for Tier 3 Neural Models (DistilBERT & Roberta) in a readable format

In [16]:
summary_rows = []
for model_key, result in all_results.items():
    summary_rows.append({
        "model_key": model_key,
        "acc_mean": result["summary"]["acc"]["mean"],
        "acc_std": result["summary"]["acc"]["std"],
        "f1_mean": result["summary"]["f1"]["mean"],
        "f1_std": result["summary"]["f1"]["std"],
        "train_time_mean": result["summary"]["train_time"]["mean"],
        "inf_time_mean": result["summary"]["inf_time"]["mean"],
    })

transformer_summary_df = pd.DataFrame(summary_rows)
display(transformer_summary_df)


,model_key,acc_mean,acc_std,f1_mean,f1_std,train_time_mean,inf_time_mean
0,20NG_distilbert-base-uncased,0.876906,0.012372,0.862592,0.010280,2432.666232,0.071024
1,TweetEval_distilbert-base-uncased,0.681548,0.015110,0.674269,0.016934,1509.762803,0.037195
2,20NG_roberta-base,0.850763,0.003774,0.830024,0.008799,980.435302,0.031737
3,TweetEval_roberta-base,0.704082,0.013255,0.689105,0.018310,723.887388,0.017393


In [17]:
# 7: Error Analysis

In [18]:
import random, numpy as np, pandas as pd, torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import warnings; warnings.filterwarnings("ignore")

print("🔍 Error Analysis: Best vs Worst Models")

# ✅ Helper: Safe mean F1 fetcher for DistilBERT/RoBERTa
def get_saved_mean_f1(dataset_key, model_name):
    key = f"{dataset_key}_{model_name.lower().replace(' ', '-')}"
    if model_name == "DistilBERT": key = f"{dataset_key}_distilbert-base-uncased"
    elif model_name == "RoBERTa": key = f"{dataset_key}_roberta-base"
    return all_results.get(key, {}).get("summary", {}).get("f1", {}).get("mean", 0.0)

# ✅ Helper: Safe prediction fetcher for DistilBERT/RoBERTa
def get_saved_preds(dataset_key, model_name, seed=42):
    key = f"{dataset_key}_{model_name.lower().replace(' ', '-')}"
    if model_name == "DistilBERT": key = f"{dataset_key}_distilbert-base-uncased"
    elif model_name == "RoBERTa": key = f"{dataset_key}_roberta-base"
    if key in all_results:
        for run in all_results[key]["raw_results"]:
            if run.get("seed") == seed and "preds" in run:
                return np.array(run["preds"])
    raise ValueError(f"Predictions not found for {key} seed {seed}.")

def get_bilstm_preds(dataset_key, seed=42):
    runs = news_bilstm["raw_results"] if dataset_key == "20NG" else tweet_bilstm["raw_results"]
    for run in runs:
        if run.get("seed") == seed and "preds" in run:
            return np.array(run["preds"])
    raise ValueError(f"BiLSTM predictions not found for {dataset_key} seed {seed}.")

def categorize_failure(text, true_label, pred_label):
    text_lower = text.lower()
    words = text.split()
    if len(words) <= 5: return "Short/Uninformative Text"
    if any(k in text_lower for k in ['http', 'www', '@', '#', 'rt:', 'via', 't.co', 'click here', 'bit.ly']):
        return "Out-of-Distribution Vocabulary"
    topic_keywords = ['computer', 'windows', 'god', 'religion', 'gun', 'israel', 'palestine', 'space', 'medical', 'car', 'bike', 'baseball', 'hockey', 'graphics', 'hardware', 'crypt']
    if sum(1 for kw in topic_keywords if kw in text_lower) >= 2 and len(words) > 15:
        return "Multi-Topic / Keyword Overlap"
    if any(k in text_lower for k in ['🙃', '🙄', '😒', '/s', 'sarcasm', 'obviously', 'great job', 'perfect timing', 'love how', 'hate that', 'yeah right']):
        return "Sarcasm/Implicit Meaning"
    if len(words) < 20 and not any(k in text_lower for k in topic_keywords):
        return "Ambiguous Ground Truth / Label Noise"
    return "Other / Mixed Context"

results_map = {
    "20NG": {"TF-IDF+LR": news_lr['summary']['f1']['mean'], "TF-IDF+SVM": news_svm['summary']['f1']['mean'],
             "BiLSTM": news_bilstm['summary']['f1']['mean'], "DistilBERT": get_saved_mean_f1("20NG", "DistilBERT"),
             "RoBERTa": get_saved_mean_f1("20NG", "RoBERTa")},
    "TweetEval": {"TF-IDF+LR": tweet_lr['summary']['f1']['mean'], "TF-IDF+SVM": tweet_svm['summary']['f1']['mean'],
                  "BiLSTM": tweet_bilstm['summary']['f1']['mean'], "DistilBERT": get_saved_mean_f1("TweetEval", "DistilBERT"),
                  "RoBERTa": get_saved_mean_f1("TweetEval", "RoBERTa")}
}

targets = {}
for ds, models in results_map.items():
    ranked = sorted(models.items(), key=lambda x: x[1], reverse=True)
    targets[ds] = {"best": ranked[0][0], "worst": ranked[-1][0],
                   "data": (news_train, news_test) if ds == "20NG" else (tweet_train, tweet_test)}

results_data = {}
for ds_name, cfg in targets.items():
    print(f"\n📊 Processing {ds_name}...")
    X_tr, y_tr = cfg["data"][0]; X_te, y_te = cfg["data"][1]
    model_preds = {}
    
    vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
    X_tr_vec = vec.fit_transform(X_tr); X_te_vec = vec.transform(X_te)
    
    if "TF-IDF+LR" in [cfg["best"], cfg["worst"]]:
        clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        clf.fit(X_tr_vec, y_tr); model_preds["TF-IDF+LR"] = clf.predict(X_te_vec)
    if "TF-IDF+SVM" in [cfg["best"], cfg["worst"]]:
        clf = SVC(kernel='linear', C=0.1, random_state=42)
        clf.fit(X_tr_vec, y_tr); model_preds["TF-IDF+SVM"] = clf.predict(X_te_vec)
        
    # ✅ FIXED: Fetch BiLSTM from correct variables
    if "BiLSTM" in [cfg["best"], cfg["worst"]]:
        model_preds["BiLSTM"] = get_bilstm_preds(ds_name, seed=42)
    if "DistilBERT" in [cfg["best"], cfg["worst"]]:
        model_preds["DistilBERT"] = get_saved_preds(ds_name, "DistilBERT", seed=42)
    if "RoBERTa" in [cfg["best"], cfg["worst"]]:
        model_preds["RoBERTa"] = get_saved_preds(ds_name, "RoBERTa", seed=42)

    random.seed(42)
    errors = {}
    for m_name, preds in model_preds.items():
        err_indices = [i for i in range(len(y_te)) if y_te[i] != preds[i]]
        sample_n = min(30, len(err_indices))
        sample_idx = random.sample(err_indices, sample_n)
        errors[m_name] = [(i, X_te[i], y_te[i], preds[i]) for i in sample_idx]
    results_data[ds_name] = {"errors": errors, "best": cfg["best"], "worst": cfg["worst"]}

print("\n" + "="*90)
print("ERROR CATEGORIZATION & OVERLAP ANALYSIS")
print("="*90)
for ds_name, data in results_data.items():
    best_name, worst_name = data["best"], data["worst"]
    if best_name not in data["errors"] or worst_name not in data["errors"]:
        print(f"⚠️ Skipped {ds_name}: Predictions missing.")
        continue
    best_df = pd.DataFrame([(i,t,y,p,categorize_failure(t,y,p)) for i,t,y,p in data["errors"][best_name]], columns=["idx","text","true","pred","category"])
    worst_df = pd.DataFrame([(i,t,y,p,categorize_failure(t,y,p)) for i,t,y,p in data["errors"][worst_name]], columns=["idx","text","true","pred","category"])
    overlap = set(best_df["idx"]) & set(worst_df["idx"])
    print(f"\n🔹 {ds_name} (Best: {best_name}, Worst: {worst_name})")
    print("Best Model Failures:\n", best_df["category"].value_counts(normalize=True).round(3))
    print("Worst Model Failures:\n", worst_df["category"].value_counts(normalize=True).round(3))
    print(f"🔗 Error Overlap: {len(overlap)} examples misclassified by both")

🔍 Error Analysis: Best vs Worst Models

📊 Processing 20NG...

📊 Processing TweetEval...

ERROR CATEGORIZATION & OVERLAP ANALYSIS

🔹 20NG (Best: TF-IDF+SVM, Worst: BiLSTM)
Best Model Failures:
 category
Other / Mixed Context                   0.433
Out-of-Distribution Vocabulary          0.267
Ambiguous Ground Truth / Label Noise    0.200
Short/Uninformative Text                0.067
Sarcasm/Implicit Meaning                0.033
Name: proportion, dtype: float64
Worst Model Failures:
 category
Other / Mixed Context                   0.500
Out-of-Distribution Vocabulary          0.333
Ambiguous Ground Truth / Label Noise    0.100
Multi-Topic / Keyword Overlap           0.067
Name: proportion, dtype: float64
🔗 Error Overlap: 3 examples misclassified by both

🔹 TweetEval (Best: RoBERTa, Worst: BiLSTM)
Best Model Failures:
 category
Out-of-Distribution Vocabulary          0.733
Ambiguous Ground Truth / Label Noise    0.133
Other / Mixed Context                   0.067
Short/Uninformative Tex

In [19]:
# 8: Per-Class F1 Analysis

In [20]:
from sklearn.metrics import classification_report
import pandas as pd

def print_per_class_f1_table(y_true, y_pred, class_names, dataset_name, model_name):
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    cls_df = pd.DataFrame(report).transpose()
    keep = class_names if not str(class_names[0]).isdigit() else [str(i) for i in range(len(class_names))]
    cls_df = cls_df.loc[[k for k in cls_df.index if k in keep]]
    
    print(f"\n{'='*75}")
    print(f"Per-Class F1 Score | Dataset: {dataset_name} | Model: {model_name}")
    print(f"{'='*75}")
    print(cls_df[['f1-score']].to_string())
    print(f"\n🏆 Best Class : {cls_df['f1-score'].idxmax()} (F1={cls_df['f1-score'].max():.4f})")
    print(f"⚠️ Worst Class: {cls_df['f1-score'].idxmin()} (F1={cls_df['f1-score'].min():.4f})")

print("⏳ Generating Per-Class F1 Tables...")

# Classical Models
print_per_class_f1_table(news_test[1], news_lr['raw_results'][0]['model'].predict(news_test[0]), news_names, "20 Newsgroups", "TF-IDF + Logistic Regression")
print_per_class_f1_table(news_test[1], news_svm['raw_results'][0]['model'].predict(news_test[0]), news_names, "20 Newsgroups", "TF-IDF + SVM")
print_per_class_f1_table(tweet_test[1], tweet_lr['raw_results'][0]['model'].predict(tweet_test[0]), tweet_names, "TweetEval Irony", "TF-IDF + Logistic Regression")
print_per_class_f1_table(tweet_test[1], tweet_svm['raw_results'][0]['model'].predict(tweet_test[0]), tweet_names, "TweetEval Irony", "TF-IDF + SVM")

# Neural & Transformer Models
for ds, X_te, y_te, names in [("20NG", news_test[0], news_test[1], news_names), 
                              ("TweetEval", tweet_test[0], tweet_test[1], tweet_names)]:
    ds_label = "20 Newsgroups" if ds == "20NG" else "TweetEval Irony"
    
    for m_name in ["BiLSTM", "DistilBERT", "RoBERTa"]:
        try:
            # ✅ Route to correct fetcher
            if m_name == "BiLSTM":
                preds = get_bilstm_preds(ds, seed=42)
            else:
                preds = get_saved_preds(ds, m_name, seed=42)
                
            print_per_class_f1_table(y_te, preds, names, ds_label, m_name)
        except Exception as e:
            print(f"\n⚠️ Skipped {m_name} on {ds_label}: {e}")

print("\n✅ Per-class F1 tables generated successfully.")

⏳ Generating Per-Class F1 Tables...

Per-Class F1 Score | Dataset: 20 Newsgroups | Model: TF-IDF + Logistic Regression
                          f1-score
alt.atheism               0.981132
comp.graphics             0.937500
comp.os.ms-windows.misc   0.965517
comp.sys.ibm.pc.hardware  0.814815
comp.sys.mac.hardware     0.965517
comp.windows.x            0.909091
misc.forsale              0.888889
rec.autos                 0.875000
rec.motorcycles           0.750000
rec.sport.baseball        0.888889
rec.sport.hockey          0.975610
sci.crypt                 0.956522
sci.electronics           0.818182
sci.med                   0.976744
sci.space                 0.750000
soc.religion.christian    0.965517
talk.politics.guns        0.758621
talk.politics.mideast     0.878049
talk.politics.misc        1.000000
talk.religion.misc        0.720000

🏆 Best Class : talk.politics.misc (F1=1.0000)
⚠️ Worst Class: talk.religion.misc (F1=0.7200)

Per-Class F1 Score | Dataset: 20 Newsgroups | Model